# Week 11 — Monday: Logistic Regression

**DATA 202 · Calvin University**

Under the Home Mortgage Disclosure Act, most U.S. mortgage lenders must report every application they receive and how it ended, and the government publishes the records. Congress passed the act in 1975 so the public could see whether lenders were serving all of their communities, after decades of redlining.

Today: **applications to borrow against a home in Kent County, Michigan**, decided in 2025. Each one ended **approved** or **denied**: a class, not a number. Weeks 6–9 classified with kNN and trees. Week 10's linear regression predicted numbers with a line.

> 🎯 **How can a linear model do classification?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~5 min | 1 · Approved or denied? A first try with a line |
| ~12 min | 2 · From a score to a probability (SLO 11A) |
| ~5 min | 3 · Fitting the S-curve (SLO 11A) |
| ~7 min | 4 · Many columns, one score (SLO 11A) |
| ~10 min | 5 · Choosing the threshold: precision and recall (SLO 11A) |
| ~2 min | 6 · More than two outcomes (SLO 11A) |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: Home Mortgage Disclosure Act (HMDA) records for 2025, Consumer Financial Protection Bureau (public domain): applications for the applicant's own single-family home</sub>

---
## 1 · Approved or Denied? A First Try With a Line · ~5 min

One row = one application. The columns we use today:

* `debt_to_income`: the share of the applicant's monthly income that would go to debt payments, the new loan included (%)
* `loan_to_value`: the loans on the home as a share of its value (%)
* `loan_amount` ($), `loan_purpose` (purchase, refinance, …), `lien` (a first mortgage or a second loan behind it), `loan_type` (conventional, FHA, …)

The file also records the applicants' race, sex and age. Today's models leave them out. Week 14 comes back to them.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, GridSearchCV
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

mortgages = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/mortgages.csv")
numeric = ["debt_to_income", "loan_to_value", "loan_amount"]
words = ["loan_purpose", "lien", "loan_type"]
complete = mortgages.dropna(subset=numeric)                  # 793 applications have a blank in these columns

X = complete[numeric + words]
y = complete["decision"]                                     # the label: approved or denied
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
print(len(X_train), "training applications |", len(X_test), "test |", f"{(y_train == 'denied').mean():.1%} denied")
complete.head()

In [ ]:
# @title Plot: the training applications by debt-to-income (run it)
import plotly.graph_objects as go

# HMDA reports the exact debt-to-income only from 36% to 49%; the file's other values stand for ranges
spans = {15: (10, 20), 25: (20, 30), 33: (30, 36), 55: (50, 60), 65: (60, 70)}
denied_share = (y_train == "denied").groupby(X_train["debt_to_income"]).mean()
lefts = [spans.get(v, (v, v + 1))[0] for v in denied_share.index]
widths = [spans.get(v, (v, v + 1))[1] - spans.get(v, (v, v + 1))[0] for v in denied_share.index]

columns = go.Figure()            # one column per debt-to-income, as wide as the range it stands for
columns.add_bar(x=lefts, y=denied_share.values, width=widths, offset=0, name="denied", marker_color="#eb6834")
columns.add_bar(x=lefts, y=1 - denied_share.values, width=widths, offset=0, name="approved", marker_color="#e3e3e3")
columns.update_traces(marker_line_color="white", marker_line_width=0.5)
columns.update_layout(barmode="stack", xaxis_range=[10, 70], xaxis_title="debt-to-income (%)",
                      yaxis_title="share of the applications", title="13,284 Training Applications by Debt-to-Income")
columns.show()

Each column holds the training applications in one debt-to-income range: **orange = denied**, gray = approved. (HMDA reports the exact debt-to-income only from 36% to 49%. The wide columns are ranges: under 20%, 20–30%, 30–36%, 50–60% and over 60%.)

> 🎯 **Predict First:** what is an applicant's chance of denial at a debt-to-income of 45%? At 55%?

<details><summary>Answer</summary>

* Up to 49%, about **1 in 10** applications is denied, whatever the debt-to-income
* Then a cliff: **44%** denied at 50–60%, **95%** above 60%. Fannie Mae, which buys a large share of U.S. mortgages, generally won't take a loan above 50%
</details>

**A first try: Week 10's line.** Turn the label into a number, 1 = denied and 0 = approved, and fit a straight line. Its guess could be read as "the chance of denial":

In [ ]:
is_denied = (y_train == "denied").astype(int)               # the label as a number: 1 = denied, 0 = approved
line = LinearRegression().fit(X_train[["debt_to_income"]], is_denied)
print(f"line: {line.intercept_:.3f} + {line.coef_[0]:.4f} × debt-to-income")

dti_range = pd.DataFrame({"debt_to_income": np.linspace(10, 70, 50)})
fig = go.Figure(columns)                                    # a copy of the columns above
fig.add_scatter(x=dti_range["debt_to_income"], y=line.predict(dti_range), mode="lines", name="Week 10's line",
                line_color="black")
fig.show()

> 💬 **Question:** Read the line as the chance of denial. What goes wrong?

<details><summary>Answer</summary>

* At a debt-to-income of 15%, the line says **−0.08**: a negative chance. Past 120% it would pass 1
* It never reaches 0.5 inside the data, so a 50% cutoff would **approve everyone**, even at 65%, where 95% are denied
* A straight line can't stay between 0 and 1, and it can't bend to follow the cliff
* The fix: keep the line, but read it as a **score**, not a chance, and squeeze the score into a probability. That is **logistic regression**
</details>

---
## 2 · From a Score to a Probability (SLO 11A) · ~12 min

**Logistic regression** is how a linear model classifies. It keeps Week 10's line, in two steps: first a **score**, then a **probability**.

### Step 1 · The score $z$

The score is a line, as in Week 10: an intercept, plus a slope times each column.

$$z = b_0 + b_1 \times \text{debt-to-income}$$

For now, take made-up numbers: $b_0 = -10$ and $b_1 = 0.2$ (Part 3 lets the data choose them). Applicant B has a debt-to-income of 50%:

$$z = -10 + 0.2 \times 50 = -10 + 10 = 0$$

> 📄 **Handout 1a, question 1.** Compute the scores of applicants A (debt-to-income 30%) and C (70%).

In [ ]:
b0, b1 = -10, 0.2                                     # made-up numbers for now: Part 3 fits them to the data
applicants = pd.DataFrame({"debt_to_income": [30, 50, 70]}, index=["A", "B", "C"])
applicants["score z"] = b0 + b1 * applicants["debt_to_income"]
applicants

<details><summary>Answer</summary>

* **A:** −10 + 0.2 × 30 = **−4** · **B:** **0** · **C:** −10 + 0.2 × 70 = **+4**
* Each extra point of debt-to-income adds $b_1$ = 0.2 to the score. A positive score leans *denied*, a negative one leans *approved*, and 0 is undecided
* The score can be **any number**: at a debt-to-income of 100%, $z$ = 10. So a score is not a probability yet: a "probability" of 10 means nothing
* With more columns, each one adds its own term: $z = b_0 + b_1 \times \text{debt-to-income} + b_2 \times \text{loan-to-value} + \dots$ (Part 4)
</details>

### Step 2 · Squeeze the score into a probability

The S-shaped **logistic function** turns any score into a number between 0 and 1:

$$p = \dfrac{1}{1 + e^{-z}}$$

| score $z$ | −5 | −2 | 0 | 2 | 5 |
|---|---|---|---|---|---|
| probability $p$ | 0.01 | 0.12 | 0.50 | 0.88 | 0.99 |

* A big negative score → $p$ near 0. A big positive score → $p$ near 1. A score of 0 → exactly 0.5
* Any score, however big, lands between 0 and 1

In [ ]:
z = np.linspace(-6, 6, 200)
px.line(x=z, y=1 / (1 + np.exp(-z)), labels={"x": "score z", "y": "probability p"}, title="The Logistic Function").show()

> 📄 **Handout 1a, questions 2–4.** Mark A, B and C on the curve. Raise each score by 1: whose probability moves the most? 🗣️ Then talk with a neighbor.

<details><summary>Answer</summary>

* **A** (z = −4): 0.02 → 0.05 · **B** (z = 0): 0.50 → 0.73 · **C** (z = +4): 0.98 → 0.99
* The same +1 moves the **borderline** applicant the most. The curve is steepest at $p$ = 0.5 and nearly flat at both ends
* For a clear case, a small change in the data changes nothing. For a borderline case, a few dollars of debt, a rounding or a typo can flip the decision
</details>

$b_0$ and $b_1$ shape the curve. Four curves for the chance of denial:

In [ ]:
# @title Four S-curves for Handout 1b (run it; the code holds the answers)
dti = np.linspace(0, 80, 200)
curves = pd.DataFrame({"debt_to_income": dti})
for number, (b0, b1) in {"1": (-6, 0.2), "2": (-25, 0.5), "3": (10, -0.2), "4": (-10, 0.2)}.items():
    curves[f"curve {number}"] = 1 / (1 + np.exp(-(b0 + b1 * dti)))
px.line(curves, x="debt_to_income", y=["curve 1", "curve 2", "curve 3", "curve 4"],
        labels={"debt_to_income": "debt-to-income (%)", "value": "P(denied)", "variable": ""},
        title="Four S-Curves: Which Score Made Each One?").show()

> 📄 **Handout 1b — Four S-curves.** Match each score to its curve. 🗣️ Which curve looks most like the rule "deny above 50%"?

<details><summary>Answer</summary>

* Each curve crosses 50% where its score is 0: at debt-to-income = $-b_0 \div b_1$
* (a) $z = -10 + 0.2 \times \text{DTI}$, Step 1's score: 50% at 50 → **curve 4**
* (b) $z = -25 + 0.5 \times \text{DTI}$: also 50% at 50, with a bigger slope, so steeper → **curve 2**
* (c) $z = -6 + 0.2 \times \text{DTI}$: 50% at 30 → **curve 1**
* (d) $z = 10 - 0.2 \times \text{DTI}$: a negative slope, so the chance *falls* → **curve 3**
* **$b_1$ sets the steepness and the direction. $b_0$ slides the curve left or right**
* Curve 2 is closest to the rule. Make the slope bigger and bigger, and the S becomes a step: a hard rule is an S-curve with an endless slope
</details>

---
## 3 · Fitting the S-Curve (SLO 11A) · ~5 min

`LogisticRegression` chooses $b_0$ and $b_1$: the curve under which the training decisions are most likely.

In [ ]:
logit = LogisticRegression().fit(X_train[["debt_to_income"]], y_train)
b0, b1 = logit.intercept_[0], logit.coef_[0][0]
print(f"z = {b0:.2f} + {b1:.3f} × debt-to-income | 50/50 at {-b0 / b1:.1f}%")

grid = pd.DataFrame({"debt_to_income": np.linspace(10, 70, 200)})
fig = go.Figure(columns)                                    # Part 1's columns again
# predict_proba gives one column per class, in the order of logit.classes_: approved, then denied
fig.add_scatter(x=grid["debt_to_income"], y=logit.predict_proba(grid)[:, 1], mode="lines", name="P(denied)",
                line_color="#1f4e99", line_width=3)
fig.update_layout(title="Logistic Regression: the Model's Chance of Denial", yaxis_title="chance of denial")
fig.show()

> 💬 **Question:** Where does this model say *denied*? Which applications does it get wrong?

<details><summary>Answer</summary>

* $z = -5.47 + 0.090 \times \text{DTI}$. The curve crosses 0.5 at **60.7%**, so the model says *denied* only for the applicants above 60%
* **Wrong:** all the orange left of 60%, **1,439** of the 2,016 denials, and the gray in the last column, **33** applications
* The orange makes a staircase: flat at about 10%, then up at 50% and again at 60%. One S-curve can't follow it: too high in the 40s, too low above 50%
* More than half of the denials come with a debt-to-income under 50%: for credit history, missing paperwork, the home's value. One column can't see those reasons
* **Reading the slope:** each +10 points of debt-to-income multiplies the **odds** of denial, $p \div (1-p)$, by $e^{10 \times 0.090} \approx 2.5$
</details>

---
## 4 · Many Columns, One Score (SLO 11A) · ~7 min

Same two steps, with one score from every column: $z = b_0 + b_1 \times \text{debt-to-income} + b_2 \times \text{loan-to-value} + \dots$, then the S-curve. The numbers are scaled and the words one-hot encoded, as in Week 9.

* A **cash-out refinance** replaces a mortgage with a bigger one and pays the difference out in cash
* A **second lien** is a second loan on a home that already has a mortgage, such as a home-equity loan
* An **FHA loan** is insured by the federal government, for borrowers with smaller down payments or lower credit scores

> 📄 **Handout 2 — Many columns, one score.** Before the next cell runs, guess the sign of each slope.

In [ ]:
model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), numeric),
        ("words", OneHotEncoder(drop="first"), words),        # baselines: cash-out refinance, first lien, FHA
    ])),
    ("logit", LogisticRegression(max_iter=5000)),
])
print(f"5-fold CV accuracy: {cross_val_score(model, X_train, y_train, cv=5).mean():.3f}"
      f" | approve everyone: {(y_train == 'approved').mean():.3f}")

model.fit(X_train, y_train)
slopes = pd.Series(model[-1].coef_[0], index=model[0].get_feature_names_out()).round(2)
slopes.index = slopes.index.str.replace("num__|words__", "", regex=True)     # drop the step-name prefixes
slopes.sort_values()

<details><summary>Answer</summary>

* `debt_to_income` **+1.03** per standard deviation: the strongest push toward denial
* `loan_amount` **−0.32**: bigger loans are denied *less*. Bigger loans tend to go to applicants with more income and more house
* `loan_purpose_purchase` **−1.42**: the odds of denial × $e^{-1.42} \approx 0.24$, against a cash-out refinance. Buying a home is the safest kind of application here
* `lien_second` **−0.23**: yet second liens are denied about twice as often (21% against 12%). Next to purpose and type, the sign flips: Week 10's "a slope depends on the other columns"
* `loan_type_conventional` **−0.59**: FHA loans are denied more, and FHA serves borrowers with weaker credit
* 🗣️ The lender saw each applicant's **credit score**. This file doesn't have it, so some slopes may stand in for it: the loan type, the purpose, the amount
* Accuracy: **88.4%**, against **84.8%** for approving everyone. At the usual 50% threshold the model rarely says *denied*: Part 5 is about that
</details>

`LogisticRegression` has Week 10's Ridge penalty built in. Its knob is `C` = 1 ÷ alpha, so a **small C = a big penalty**:

In [ ]:
Cs = [0.0001, 0.001, 0.01, 1, 100]
search = GridSearchCV(model, {"logit__C": Cs}, cv=5, return_train_score=True).fit(X_train, y_train)
pd.DataFrame({"folds it learned from": search.cv_results_["mean_train_score"],
              "held-out fold": search.cv_results_["mean_test_score"]}, index=pd.Index(Cs, name="C")).round(3)

<details><summary>Answer</summary>

* **C = 0.0001:** the penalty flattens every slope, and the model approves everyone (84.8%): **underfitting**
* From **C = 0.01** on, about 88.5% on both: no gap, so **no overfitting**. 11 slopes are too little freedom to memorize 13,284 applications
</details>

---
## 5 · Choosing the Threshold: Precision and Recall (SLO 11A) · ~10 min

`predict` flags an application *denied* when its chance is at least **0.5**. That 0.5 is the **threshold**, and it is a choice. Week 6's tools show what the choice changes. A **positive** here is a *denied* application:

| | the lender denied it | the lender approved it |
|---|---|---|
| **the model flags it** (chance ≥ threshold) | TP: true positive | FP: false positive |
| **the model doesn't** | FN: false negative | TN: true negative |

* **Precision** = TP ÷ (TP + FP): of the applications the model flags, the share the lender denied
* **Recall** = TP ÷ (TP + FN): of the applications the lender denied, the share the model flags

Each training application gets its chance from a model that didn't learn from it, fit on the other 4 folds:

In [ ]:
chance = cross_val_predict(model, X_train, y_train, cv=5, method="predict_proba")[:, 1]    # P(denied)

ten = pd.DataFrame({"chance of denial": chance, "lender's decision": y_train}, index=X_train.index)
ten = ten.loc[[15085, 3357, 15992, 956, 3746, 13129, 2383, 14253, 6724, 1311]]      # ten applications, for the handout
ten.index = range(1, 11)
ten.round(2)

> 📄 **Handout 3 — Choose the threshold.** Count TP, FP, FN and TN at a threshold of 50%, then at 20%, and compute precision and recall. 🗣️ Which threshold would you choose?

<details><summary>Answer</summary>

* **50%:** the model flags 1, 2 and 3. TP = 2 (1, 2), FP = 1 (3), FN = 2 (4, 7), TN = 5. Precision 2 ÷ 3 = **0.67**, recall 2 ÷ 4 = **0.50**
* **20%:** the model flags 1 to 6. TP = 3 (1, 2, 4), FP = 3 (3, 5, 6), FN = 1 (7), TN = 3. Precision 3 ÷ 6 = **0.50**, recall 3 ÷ 4 = **0.75**
* A lower threshold flags more applications: it catches more of the denials (**recall up**), but more of what it flags was approved (**precision down**)
</details>

The same counts for all 13,284 training applications:

In [ ]:
denied = (y_train == "denied").values
rows = []
for threshold in [0.6, 0.5, 0.4, 0.3, 0.2, 0.1]:
    flagged = chance >= threshold
    tp, fp = (flagged & denied).sum(), (flagged & ~denied).sum()
    fn, tn = (~flagged & denied).sum(), (~flagged & ~denied).sum()
    rows.append({"threshold": threshold, "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                 "precision": tp / (tp + fp), "recall": tp / (tp + fn), "accuracy": (tp + tn) / len(denied)})
pd.DataFrame(rows).round(3)

<details><summary>Answer</summary>

* **At 50%:** precision 0.81, recall 0.31. When the model flags an application it's usually right, but it misses 7 of every 10 denials
* **At 10%:** recall 0.80, precision 0.27. It catches most denials, but most of what it flags, the lender approved
* **Accuracy** is highest near 50% (0.884) and falls to 0.64 at 10%. With 85% of applications approved, accuracy rewards saying *approved*, and it hides the trade-off
* No threshold is "correct": it depends on which mistake costs more, and to whom. Week 12 comes back to this
</details>

> "Organizations apply labeling and scoring methods to slice people into groups and ranks suited to their particular ends. […] Instead they tend to be behavioral and probabilistic, predicting the likelihood that people will do or like certain kinds of things, or the position they may reach on a particular scale."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** A lender uses this model to decide. Who should choose the threshold, and what should they weigh? What does a turned-down applicant get to see?

<details><summary>Answer</summary>

* A **false positive** turns away someone the lender would have approved: the applicant pays. A **false negative** approves someone the lender would have denied: the lender takes the risk. Each side would pick a different threshold
* The threshold is a choice, not a fact of the data. It serves someone's "particular ends", so it matters whose ends count
* The label hides the number: 0.49 and 0.03 both become *approved*. And by law (the Equal Credit Opportunity Act), a denied applicant must be told the main reasons. "Your chance was 0.52" is not a reason
</details>

> 📌 **One split:** every score today comes from cross-validation on the training applications. The test set stays locked until Week 14, when we ask how this model treats different groups of applicants.

---
## 6 · More Than Two Outcomes (SLO 11A) · ~2 min

HMDA records more than *approved* and *denied*: some applicants withdraw, and some files are closed as incomplete. With three or more outcomes, `LogisticRegression`:

1. Gives **each outcome its own score** $z$ (its own line)
2. Turns the scores into probabilities that **add up to 1** (the *softmax*: $e^{z}$ for each, divided by the total)
3. Predicts the outcome with the **largest** probability

Same code, `LogisticRegression().fit(X, y)`, with three names in `y`. For one application:

In [ ]:
scores = pd.Series({"approved": 1.5, "denied": 0.2, "withdrawn": -0.5})
probabilities = np.exp(scores) / np.exp(scores).sum()       # softmax
probabilities.round(2)

<details><summary>Answer</summary>

* 71% approved, 19% denied, 10% withdrawn → **approved**
* With two outcomes, this is exactly Part 2's S-curve
</details>

---
### Wrap-up

1. A linear model classifies with **logistic regression**: the line gives a score, the logistic function squeezes it into a probability between 0 and 1, and a threshold turns the probability into a class
2. $b_1$ sets the curve's steepness and direction, $b_0$ slides it. 50/50 where the score is 0
3. A slope's sign says which way the odds move; $e^{\text{slope}}$ says by how much
4. The **threshold is a choice**: lower it, and recall rises while precision falls. Accuracy hides that trade-off 📌
5. More than two outcomes: one score each, softmax, pick the largest

**Wednesday:** trees that vote: random forests and boosting.